# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
# Δ-WINDOW PROJECT — COUPLED LOGISTIC MAPS v1
# 1D ring network + backup structure
# ============================================================

!pip -q install numpy pandas matplotlib tqdm scipy


import os
import json
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from scipy.spatial import cKDTree

# ============================================================
# PATHS
# ============================================================

PROJECT_NAME = "delta_window_coupled_logistic_maps_v1"
BASE_DIR = "data/raw"
PROJECT_DIR = os.path.join(BASE_DIR, PROJECT_NAME)

CSV_DIR = os.path.join(PROJECT_DIR, "csv")
FIG_DIR = os.path.join(PROJECT_DIR, "figures")
FINAL_FIG_DIR = os.path.join(PROJECT_DIR, "final_figures")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

for d in [PROJECT_DIR, CSV_DIR, FIG_DIR, FINAL_FIG_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_ID:", RUN_ID)

# ============================================================
# HELPERS
# ============================================================

def csv_path(name):
    return os.path.join(CSV_DIR, name)

def fig_path(name):
    return os.path.join(FIG_DIR, name)

def final_fig_path(name):
    return os.path.join(FINAL_FIG_DIR, name)

def checkpoint_path(name):
    return os.path.join(CHECKPOINT_DIR, name)

# ============================================================
# GLOBAL SETTINGS
# ============================================================

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("Opening block ready.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — MAIN TEST
# 1D ring + ε sweep + sync/clusters/switching/dwell
# autosave + checkpoint + resume
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# PARAMETERS
# ============================================================

N = 200                     # number of nodes
R = 3.8                     # logistic map parameter
T = 5000                    # total iterations
DISCARD = 1000              # transient
EPS_VALUES = np.round(np.linspace(0.0, 1.0, 51), 4)

SYNC_THRESHOLD = 1e-3       # threshold for synchronized nodes
CLUSTER_THRESHOLD = 1e-2    # threshold for cluster grouping
DWELL_THRESHOLD = 1e-2      # threshold for state switching/dwell

RESULT_FILE = f"clm_v1_main_results_{RUN_ID}.csv"
CHECKPOINT_FILE = "clm_v1_main_checkpoint.csv"

result_path = csv_path(RESULT_FILE)
checkpoint_file = checkpoint_path(CHECKPOINT_FILE)

# ============================================================
# LOGISTIC MAP + RING COUPLING
# ============================================================

def f_logistic(x, r=R):
    return r * x * (1.0 - x)

def simulate_coupled_logistic_ring(
    epsilon,
    n=N,
    r=R,
    t=T,
    discard=DISCARD,
    seed=42
):
    rng = np.random.default_rng(seed)

    x = rng.random(n)
    history = []

    for step in range(t):
        fx = f_logistic(x, r)

        left = np.roll(fx, 1)
        right = np.roll(fx, -1)

        x_next = (1 - epsilon) * fx + (epsilon / 2.0) * (left + right)

        x = np.clip(x_next, 0, 1)

        if step >= discard:
            history.append(x.copy())

    return np.array(history)

# ============================================================
# METRICS
# ============================================================

def synchronization_metric(history):
    """
    Lower variance = stronger synchronization.
    sync_score = 1 / (1 + mean spatial variance)
    """
    spatial_var = np.var(history, axis=1)
    mean_var = np.mean(spatial_var)
    sync_score = 1.0 / (1.0 + mean_var)
    return mean_var, sync_score

def cluster_count_snapshot(x, threshold=CLUSTER_THRESHOLD):
    """
    Simple 1D value-based clustering on sorted node values.
    """
    xs = np.sort(x)
    diffs = np.diff(xs)

    if len(diffs) == 0:
        return 1

    return 1 + np.sum(diffs > threshold)

def cluster_metrics(history):
    counts = np.array([
        cluster_count_snapshot(x)
        for x in history
    ])

    return {
        "mean_clusters": np.mean(counts),
        "median_clusters": np.median(counts),
        "std_clusters": np.std(counts),
        "min_clusters": np.min(counts),
        "max_clusters": np.max(counts)
    }

def switching_metric(history, threshold=DWELL_THRESHOLD):
    """
    Counts how often nodes change state more than threshold between steps.
    """
    diffs = np.abs(np.diff(history, axis=0))
    switching_events = diffs > threshold

    total_switches = np.sum(switching_events)
    mean_switches_per_step = np.mean(np.sum(switching_events, axis=1))
    mean_switch_fraction = np.mean(switching_events)

    return {
        "total_switches": total_switches,
        "mean_switches_per_step": mean_switches_per_step,
        "mean_switch_fraction": mean_switch_fraction
    }

def dwell_times_binary_sync(history, threshold=SYNC_THRESHOLD):
    """
    Dwell of global synchronized / non-synchronized episodes.
    State = 1 if spatial std < threshold, else 0.
    """
    spatial_std = np.std(history, axis=1)
    states = (spatial_std < threshold).astype(int)

    dwell = []
    current = states[0]
    length = 1

    for s in states[1:]:
        if s == current:
            length += 1
        else:
            dwell.append((current, length))
            current = s
            length = 1

    dwell.append((current, length))

    dwell_df = pd.DataFrame(dwell, columns=["state", "dwell_length"])

    return dwell_df

def compute_metrics_for_epsilon(epsilon):
    history = simulate_coupled_logistic_ring(
        epsilon=epsilon,
        seed=SEED
    )

    mean_var, sync_score = synchronization_metric(history)
    clust = cluster_metrics(history)
    sw = switching_metric(history)
    dwell_df = dwell_times_binary_sync(history)

    sync_dwell = dwell_df[dwell_df["state"] == 1]["dwell_length"]
    nonsync_dwell = dwell_df[dwell_df["state"] == 0]["dwell_length"]

    row = {
        "epsilon": epsilon,
        "N": N,
        "R": R,
        "T": T,
        "DISCARD": DISCARD,

        "mean_spatial_var": mean_var,
        "sync_score": sync_score,

        **clust,
        **sw,

        "n_dwell_segments": len(dwell_df),
        "mean_sync_dwell": sync_dwell.mean() if len(sync_dwell) else 0,
        "max_sync_dwell": sync_dwell.max() if len(sync_dwell) else 0,
        "mean_nonsync_dwell": nonsync_dwell.mean() if len(nonsync_dwell) else 0,
        "max_nonsync_dwell": nonsync_dwell.max() if len(nonsync_dwell) else 0,
    }

    return row

# ============================================================
# RESUME LOGIC
# ============================================================

if os.path.exists(checkpoint_file):
    df_done = pd.read_csv(checkpoint_file)
    done_eps = set(np.round(df_done["epsilon"].values, 4))
    rows = df_done.to_dict("records")

    print("Loaded checkpoint:")
    print(checkpoint_file)
    print("Completed epsilons:", len(done_eps))

else:
    rows = []
    done_eps = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for eps in tqdm(EPS_VALUES, desc="epsilon loop"):

    eps_key = round(float(eps), 4)

    if eps_key in done_eps:
        continue

    row = compute_metrics_for_epsilon(eps_key)

    rows.append(row)
    done_eps.add(eps_key)

    partial_df = pd.DataFrame(rows).sort_values("epsilon").reset_index(drop=True)

    partial_df.to_csv(checkpoint_file, index=False)
    partial_df.to_csv(result_path, index=False)

    print(f"Saved thresholdress after epsilon={eps_key}")

# ============================================================
# FINAL SAVE
# ============================================================

main_df = pd.DataFrame(rows).sort_values("epsilon").reset_index(drop=True)

main_df.to_csv(result_path, index=False)
main_df.to_csv(checkpoint_file, index=False)

print("\nCoupled Logistic Maps v1 main test completed.")
print("Rows:", len(main_df))
print("Saved:", result_path)
print("Checkpoint:", checkpoint_file)

display(main_df.head())

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — DIAGNOSTIC PLOTS
# epsilon vs clusters / switches / sync_score
# reads latest main results + saves figures
# ============================================================

import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# FIND LATEST MAIN RESULTS
# ============================================================

latest_file = sorted(
    glob.glob(csv_path("clm_v1_main_results_*.csv"))
)[-1]

df = pd.read_csv(latest_file)

print("Loaded:")
print(latest_file)
print("Rows:", len(df))

display(df.head())

# ============================================================
# FIGURE 1 — sync_score vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["sync_score"],
    marker="o"
)

plt.xlabel("coupling ε")
plt.ylabel("sync_score")
plt.title("Coupled Logistic Maps v1 — sync_score vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_sync_score_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 2 — mean spatial variance vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["mean_spatial_var"],
    marker="o"
)

plt.xlabel("coupling ε")
plt.ylabel("mean spatial variance")
plt.title("Coupled Logistic Maps v1 — spatial variance vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_spatial_variance_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 3 — mean clusters vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["mean_clusters"],
    marker="o",
    label="mean clusters"
)

plt.fill_between(
    df["epsilon"],
    df["mean_clusters"] - df["std_clusters"],
    df["mean_clusters"] + df["std_clusters"],
    alpha=0.25,
    label="± std"
)

plt.xlabel("coupling ε")
plt.ylabel("clusters")
plt.title("Coupled Logistic Maps v1 — mean clusters vs ε")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_mean_clusters_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 4 — median clusters vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["median_clusters"],
    marker="o"
)

plt.xlabel("coupling ε")
plt.ylabel("median clusters")
plt.title("Coupled Logistic Maps v1 — median clusters vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_median_clusters_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 5 — switching fraction vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["mean_switch_fraction"],
    marker="o"
)

plt.xlabel("coupling ε")
plt.ylabel("mean switch fraction")
plt.title("Coupled Logistic Maps v1 — switching fraction vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_switch_fraction_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 6 — switches per step vs epsilon
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    df["epsilon"],
    df["mean_switches_per_step"],
    marker="o"
)

plt.xlabel("coupling ε")
plt.ylabel("mean switches per step")
plt.title("Coupled Logistic Maps v1 — switches per step vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_switches_per_step_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FIGURE 7 — combined normalized overview
# ============================================================

plot_df = df.copy()

cols = [
    "sync_score",
    "mean_clusters",
    "mean_switch_fraction",
    "mean_spatial_var"
]

for col in cols:
    mn = plot_df[col].min()
    mx = plot_df[col].max()
    plot_df[col + "_norm"] = (plot_df[col] - mn) / (mx - mn)

plt.figure(figsize=(9,6))

plt.plot(plot_df["epsilon"], plot_df["sync_score_norm"], marker="o", label="sync_score")
plt.plot(plot_df["epsilon"], plot_df["mean_clusters_norm"], marker="o", label="mean_clusters")
plt.plot(plot_df["epsilon"], plot_df["mean_switch_fraction_norm"], marker="o", label="switch_fraction")
plt.plot(plot_df["epsilon"], plot_df["mean_spatial_var_norm"], marker="o", label="spatial_var")

plt.xlabel("coupling ε")
plt.ylabel("normalized value")
plt.title("Coupled Logistic Maps v1 — normalized overview")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_normalized_overview_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("Diagnostic plots completed.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — CLUSTER-STATE DWELL TEST
# cluster dwell + switching + memory + plots + CSV
# autosave / checkpoint / resume
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# PARAMETERS
# ============================================================

N = 200
R = 3.8
T = 5000
DISCARD = 1000

EPS_VALUES = np.round(np.linspace(0.0, 1.0, 51), 4)

CLUSTER_THRESHOLD = 1e-2
STATE_TOL = 1          # dwell state = cluster count within +/- 1
N_BINS = 20            # optional coarse fingerprint

RESULT_FILE = f"clm_v1_cluster_dwell_{RUN_ID}.csv"
SUMMARY_FILE = f"clm_v1_cluster_dwell_summary_{RUN_ID}.csv"
CHECKPOINT_FILE = "clm_v1_cluster_dwell_checkpoint.csv"

result_path = csv_path(RESULT_FILE)
summary_path = csv_path(SUMMARY_FILE)
checkpoint_file = checkpoint_path(CHECKPOINT_FILE)

# ============================================================
# LOGISTIC MAP + RING COUPLING
# ============================================================

def f_logistic(x, r=R):
    return r * x * (1.0 - x)

def simulate_coupled_logistic_ring(
    epsilon,
    n=N,
    r=R,
    t=T,
    discard=DISCARD,
    seed=42
):
    rng = np.random.default_rng(seed)
    x = rng.random(n)
    history = []

    for step in range(t):
        fx = f_logistic(x, r)

        left = np.roll(fx, 1)
        right = np.roll(fx, -1)

        x_next = (1 - epsilon) * fx + (epsilon / 2.0) * (left + right)
        x = np.clip(x_next, 0, 1)

        if step >= discard:
            history.append(x.copy())

    return np.array(history)

# ============================================================
# CLUSTER STATE
# ============================================================

def cluster_count_snapshot(x, threshold=CLUSTER_THRESHOLD):
    xs = np.sort(x)
    diffs = np.diff(xs)

    if len(diffs) == 0:
        return 1

    return int(1 + np.sum(diffs > threshold))

def cluster_count_series(history):
    return np.array([
        cluster_count_snapshot(x)
        for x in history
    ])

def coarse_cluster_state(count, tol=STATE_TOL):
    """
    Coarse cluster-state label.
    Example: tol=1 groups 14,15 into nearby bins.
    """
    return int(np.round(count / tol) * tol)

def dwell_from_states(states):
    dwell_rows = []

    current = states[0]
    length = 1

    for s in states[1:]:
        if s == current:
            length += 1
        else:
            dwell_rows.append({
                "state": current,
                "dwell_length": length
            })
            current = s
            length = 1

    dwell_rows.append({
        "state": current,
        "dwell_length": length
    })

    return pd.DataFrame(dwell_rows)

# ============================================================
# METRICS FOR ONE EPSILON
# ============================================================

def compute_cluster_dwell_for_epsilon(epsilon):

    history = simulate_coupled_logistic_ring(
        epsilon=epsilon,
        seed=SEED
    )

    counts = cluster_count_series(history)

    states = np.array([
        coarse_cluster_state(c)
        for c in counts
    ])

    dwell_df = dwell_from_states(states)

    dwell_lengths = dwell_df["dwell_length"].values

    switching_count = len(dwell_df) - 1
    switching_rate = switching_count / len(states)

    dominant_state = dwell_df["state"].value_counts().idxmax()
    dominant_fraction = np.mean(states == dominant_state)

    row = {
        "epsilon": epsilon,

        "mean_cluster_count": np.mean(counts),
        "median_cluster_count": np.median(counts),
        "std_cluster_count": np.std(counts),
        "min_cluster_count": np.min(counts),
        "max_cluster_count": np.max(counts),

        "mean_dwell": np.mean(dwell_lengths),
        "median_dwell": np.median(dwell_lengths),
        "std_dwell": np.std(dwell_lengths),
        "max_dwell": np.max(dwell_lengths),

        "switching_count": switching_count,
        "switching_rate": switching_rate,

        "dominant_cluster_state": dominant_state,
        "dominant_fraction": dominant_fraction,

        "n_segments": len(dwell_df),
        "n_steps": len(states),

        "cluster_threshold": CLUSTER_THRESHOLD,
        "state_tol": STATE_TOL
    }

    raw_rows = []

    for i, r in dwell_df.reset_index(drop=True).iterrows():
        raw_rows.append({
            "epsilon": epsilon,
            "segment_id": i,
            "state": r["state"],
            "dwell_length": r["dwell_length"]
        })

    return row, raw_rows

# ============================================================
# RESUME LOGIC
# ============================================================

if os.path.exists(checkpoint_file):
    summary_done = pd.read_csv(checkpoint_file)
    done_eps = set(np.round(summary_done["epsilon"].values, 4))
    summary_rows = summary_done.to_dict("records")

    if os.path.exists(result_path):
        raw_rows = pd.read_csv(result_path).to_dict("records")
    else:
        raw_rows = []

    print("Loaded checkpoint:", checkpoint_file)
    print("Completed epsilons:", len(done_eps))

else:
    done_eps = set()
    summary_rows = []
    raw_rows = []
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for eps in tqdm(EPS_VALUES, desc="epsilon loop"):

    eps_key = round(float(eps), 4)

    if eps_key in done_eps:
        continue

    row, eps_raw_rows = compute_cluster_dwell_for_epsilon(eps_key)

    summary_rows.append(row)
    raw_rows.extend(eps_raw_rows)
    done_eps.add(eps_key)

    partial_summary = pd.DataFrame(summary_rows).sort_values("epsilon").reset_index(drop=True)
    partial_raw = pd.DataFrame(raw_rows).sort_values(["epsilon", "segment_id"]).reset_index(drop=True)

    partial_summary.to_csv(checkpoint_file, index=False)
    partial_summary.to_csv(summary_path, index=False)
    partial_raw.to_csv(result_path, index=False)

    print(f"Saved cluster-dwell thresholdress after epsilon={eps_key}")

# ============================================================
# FINAL SAVE
# ============================================================

summary_df = pd.DataFrame(summary_rows).sort_values("epsilon").reset_index(drop=True)
raw_df = pd.DataFrame(raw_rows).sort_values(["epsilon", "segment_id"]).reset_index(drop=True)

summary_df.to_csv(summary_path, index=False)
summary_df.to_csv(checkpoint_file, index=False)
raw_df.to_csv(result_path, index=False)

print("\nCluster-state dwell test completed.")
print("Summary rows:", len(summary_df))
print("Raw dwell rows:", len(raw_df))
print("Saved summary:", summary_path)
print("Saved raw:", result_path)

display(summary_df.head())

# ============================================================
# PLOTS
# ============================================================

# ------------------------------------------------------------
# Mean dwell vs epsilon
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["epsilon"], summary_df["mean_dwell"], marker="o")
plt.xlabel("coupling ε")
plt.ylabel("mean cluster-state dwell")
plt.title("CLM v1 — mean cluster-state dwell vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_mean_dwell_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# Max dwell vs epsilon
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["epsilon"], summary_df["max_dwell"], marker="o")
plt.xlabel("coupling ε")
plt.ylabel("max cluster-state dwell")
plt.title("CLM v1 — max cluster-state dwell vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_max_dwell_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# Switching rate vs epsilon
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["epsilon"], summary_df["switching_rate"], marker="o")
plt.xlabel("coupling ε")
plt.ylabel("cluster-state switching rate")
plt.title("CLM v1 — cluster-state switching rate vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_switching_rate_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# Dominant fraction vs epsilon
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["epsilon"], summary_df["dominant_fraction"], marker="o")
plt.xlabel("coupling ε")
plt.ylabel("dominant state fraction")
plt.title("CLM v1 — dominant cluster-state fraction vs ε")
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_dominant_fraction_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# Cluster count vs epsilon
# ------------------------------------------------------------

plt.figure(figsize=(8,5))
plt.plot(summary_df["epsilon"], summary_df["mean_cluster_count"], marker="o", label="mean")
plt.fill_between(
    summary_df["epsilon"],
    summary_df["mean_cluster_count"] - summary_df["std_cluster_count"],
    summary_df["mean_cluster_count"] + summary_df["std_cluster_count"],
    alpha=0.25,
    label="± std"
)
plt.xlabel("coupling ε")
plt.ylabel("cluster count")
plt.title("CLM v1 — cluster count vs ε")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_count_vs_epsilon_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# Normalized overview
# ------------------------------------------------------------

plot_df = summary_df.copy()

cols = [
    "mean_dwell",
    "max_dwell",
    "switching_rate",
    "dominant_fraction",
    "mean_cluster_count"
]

for col in cols:
    mn = plot_df[col].min()
    mx = plot_df[col].max()
    plot_df[col + "_norm"] = (plot_df[col] - mn) / (mx - mn) if mx > mn else 0

plt.figure(figsize=(9,6))

for col in cols:
    plt.plot(
        plot_df["epsilon"],
        plot_df[col + "_norm"],
        marker="o",
        label=col
    )

plt.xlabel("coupling ε")
plt.ylabel("normalized value")
plt.title("CLM v1 — cluster dwell normalized overview")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_cluster_dwell_normalized_overview_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("Cluster-state dwell plots completed.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — EPSILON REORGANIZATION WINDOW
# derivative peaks + Δ-window-like reorganization score
# CSV + plots
# ============================================================

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD LATEST FILES
# ============================================================

main_file = sorted(glob.glob(csv_path("clm_v1_main_results_*.csv")))[-1]
cluster_dwell_file = sorted(glob.glob(csv_path("clm_v1_cluster_dwell_summary_*.csv")))[-1]

main_df = pd.read_csv(main_file)
dwell_df = pd.read_csv(cluster_dwell_file)

print("Loaded main:", main_file)
print("Loaded dwell:", cluster_dwell_file)

# ============================================================
# MERGE
# ============================================================

df = main_df.merge(
    dwell_df,
    on="epsilon",
    how="left",
    suffixes=("_main", "_cluster")
)

df = df.sort_values("epsilon").reset_index(drop=True)

# ============================================================
# METRICS TO DIFFERENTIATE
# ============================================================

metrics = [
    "sync_score",
    "mean_spatial_var",
    "mean_clusters",
    "mean_switch_fraction",
    "mean_dwell",
    "max_dwell",
    "switching_rate",
    "dominant_fraction",
    "mean_cluster_count"
]

eps = df["epsilon"].values

# ============================================================
# DERIVATIVES
# ============================================================

for col in metrics:
    y = df[col].values.astype(float)
    dy = np.gradient(y, eps)
    df[f"d_{col}_dEps"] = dy
    df[f"abs_d_{col}_dEps"] = np.abs(dy)

# ============================================================
# NORMALIZED REORGANIZATION SCORE
# ============================================================

score_cols = [
    "abs_d_sync_score_dEps",
    "abs_d_mean_spatial_var_dEps",
    "abs_d_mean_clusters_dEps",
    "abs_d_mean_switch_fraction_dEps",
    "abs_d_mean_dwell_dEps",
    "abs_d_switching_rate_dEps",
    "abs_d_dominant_fraction_dEps",
    "abs_d_mean_cluster_count_dEps"
]

for col in score_cols:
    mn = df[col].min()
    mx = df[col].max()
    if mx > mn:
        df[col + "_norm"] = (df[col] - mn) / (mx - mn)
    else:
        df[col + "_norm"] = 0.0

norm_cols = [c + "_norm" for c in score_cols]

df["reorganization_score"] = df[norm_cols].mean(axis=1)

# ============================================================
# FIND PEAK WINDOW
# ============================================================

idx_peak = df["reorganization_score"].idxmax()

eps_star = df.loc[idx_peak, "epsilon"]
score_peak = df.loc[idx_peak, "reorganization_score"]

# window = score >= 75% of peak
threshold = 0.75 * score_peak

window_df = df[df["reorganization_score"] >= threshold]

eps_window_min = window_df["epsilon"].min()
eps_window_max = window_df["epsilon"].max()

summary = pd.DataFrame([{
    "epsilon_star": eps_star,
    "reorganization_score_peak": score_peak,
    "window_threshold_75pct": threshold,
    "epsilon_window_min": eps_window_min,
    "epsilon_window_max": eps_window_max,
    "n_points_in_window": len(window_df)
}])

# ============================================================
# SAVE CSV
# ============================================================

out_enriched = csv_path(f"clm_v1_reorganization_enriched_{RUN_ID}.csv")
out_summary = csv_path(f"clm_v1_reorganization_summary_{RUN_ID}.csv")

df.to_csv(out_enriched, index=False)
summary.to_csv(out_summary, index=False)

print("\nSaved enriched:")
print(out_enriched)

print("\nSaved summary:")
print(out_summary)

display(summary)

# ============================================================
# PLOT 1 — REORGANIZATION SCORE
# ============================================================

plt.figure(figsize=(9,5))

plt.plot(
    df["epsilon"],
    df["reorganization_score"],
    marker="o",
    label="reorganization score"
)

plt.axvline(eps_star, linestyle="--", label=f"ε*={eps_star:.3f}")
plt.axhline(threshold, linestyle=":", label="75% peak threshold")

plt.axvspan(
    eps_window_min,
    eps_window_max,
    alpha=0.2,
    label="reorganization window"
)

plt.xlabel("coupling ε")
plt.ylabel("score")
plt.title("CLM v1 — ε-window of reorganization")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_reorganization_score_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 2 — DERIVATIVE COMPONENTS NORMALIZED
# ============================================================

plt.figure(figsize=(10,6))

component_cols = [
    "abs_d_sync_score_dEps_norm",
    "abs_d_mean_clusters_dEps_norm",
    "abs_d_mean_dwell_dEps_norm",
    "abs_d_switching_rate_dEps_norm",
    "abs_d_dominant_fraction_dEps_norm",
    "abs_d_mean_cluster_count_dEps_norm"
]

for col in component_cols:
    plt.plot(
        df["epsilon"],
        df[col],
        marker="o",
        label=col.replace("abs_d_", "").replace("_dEps_norm", "")
    )

plt.axvspan(eps_window_min, eps_window_max, alpha=0.15)

plt.xlabel("coupling ε")
plt.ylabel("normalized derivative")
plt.title("CLM v1 — normalized derivative components")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_reorganization_components_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 3 — RAW KEY METRICS WITH WINDOW
# ============================================================

plot_df = df.copy()

raw_cols = [
    "sync_score",
    "mean_clusters",
    "mean_dwell",
    "switching_rate",
    "dominant_fraction"
]

for col in raw_cols:
    mn = plot_df[col].min()
    mx = plot_df[col].max()
    plot_df[col + "_norm"] = (plot_df[col] - mn) / (mx - mn) if mx > mn else 0.0

plt.figure(figsize=(10,6))

for col in raw_cols:
    plt.plot(
        plot_df["epsilon"],
        plot_df[col + "_norm"],
        marker="o",
        label=col
    )

plt.axvspan(eps_window_min, eps_window_max, alpha=0.15, label="ε-window")

plt.xlabel("coupling ε")
plt.ylabel("normalized value")
plt.title("CLM v1 — key metrics with reorganization window")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_key_metrics_with_window_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 4 — HEATMAP-LIKE STRIPE OF SCORE
# ============================================================

plt.figure(figsize=(10,2.5))

plt.imshow(
    df["reorganization_score"].values.reshape(1, -1),
    aspect="auto",
    extent=[df["epsilon"].min(), df["epsilon"].max(), 0, 1]
)

plt.colorbar(label="reorganization score")

plt.axvline(eps_star, linestyle="--")

plt.xlabel("coupling ε")
plt.yticks([])
plt.title("CLM v1 — reorganization score stripe")

plt.tight_layout()

fname = fig_path(f"clm_v1_reorganization_score_stripe_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\nReorganization window pipeline completed.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — SHUFFLE CONTROL FOR REORG WINDOW
# shuffle epsilon-order of metrics, recompute reorganization peak
# CSV + plots + autosave
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# LOAD ENRICHED REORGANIZATION DATA
# ============================================================

enriched_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv"))
)[-1]

df = pd.read_csv(enriched_file).sort_values("epsilon").reset_index(drop=True)

print("Loaded:")
print(enriched_file)
print("Rows:", len(df))

# ============================================================
# PARAMETERS
# ============================================================

N_SHUFFLES = 500

METRICS = [
    "sync_score",
    "mean_spatial_var",
    "mean_clusters",
    "mean_switch_fraction",
    "mean_dwell",
    "max_dwell",
    "switching_rate",
    "dominant_fraction",
    "mean_cluster_count"
]

RESULT_FILE = f"clm_v1_reorganization_shuffle_control_{RUN_ID}.csv"
SUMMARY_FILE = f"clm_v1_reorganization_shuffle_summary_{RUN_ID}.csv"
CHECKPOINT_FILE = "clm_v1_reorganization_shuffle_checkpoint.csv"

result_file = csv_path(RESULT_FILE)
summary_file = csv_path(SUMMARY_FILE)
checkpoint_file = checkpoint_path(CHECKPOINT_FILE)

eps = df["epsilon"].values

# ============================================================
# SCORE FUNCTION
# ============================================================

def compute_reorganization_score_from_metrics(input_df, metrics=METRICS):
    temp = input_df.copy().sort_values("epsilon").reset_index(drop=True)
    eps_vals = temp["epsilon"].values

    score_cols = []

    for col in metrics:
        y = temp[col].values.astype(float)
        dy = np.gradient(y, eps_vals)
        abs_col = f"abs_d_{col}_dEps"
        temp[abs_col] = np.abs(dy)

        mn = temp[abs_col].min()
        mx = temp[abs_col].max()

        norm_col = abs_col + "_norm"

        if mx > mn:
            temp[norm_col] = (temp[abs_col] - mn) / (mx - mn)
        else:
            temp[norm_col] = 0.0

        score_cols.append(norm_col)

    temp["reorganization_score"] = temp[score_cols].mean(axis=1)

    idx_peak = temp["reorganization_score"].idxmax()

    eps_star = temp.loc[idx_peak, "epsilon"]
    peak_score = temp.loc[idx_peak, "reorganization_score"]

    return eps_star, peak_score, temp["reorganization_score"].values

# ============================================================
# ORIGINAL SCORE
# ============================================================

orig_eps_star, orig_peak, orig_score = compute_reorganization_score_from_metrics(df)

print("Original epsilon_star:", orig_eps_star)
print("Original peak:", orig_peak)

# ============================================================
# RESUME LOGIC
# ============================================================

if os.path.exists(checkpoint_file):
    shuffle_df = pd.read_csv(checkpoint_file)
    done_ids = set(shuffle_df["shuffle_id"].values)
    rows = shuffle_df.to_dict("records")

    print("Loaded checkpoint:", checkpoint_file)
    print("Completed shuffles:", len(done_ids))

else:
    done_ids = set()
    rows = []
    print("No checkpoint found. Starting fresh.")

# ============================================================
# SHUFFLE LOOP
# ============================================================

for s in tqdm(range(N_SHUFFLES), desc="shuffle loop"):

    if s in done_ids:
        continue

    shuf = df[["epsilon"] + METRICS].copy()

    # shuffle each metric independently across epsilon
    for col in METRICS:
        shuf[col] = np.random.permutation(shuf[col].values)

    eps_star_s, peak_s, score_s = compute_reorganization_score_from_metrics(shuf)

    rows.append({
        "shuffle_id": s,
        "epsilon_star": eps_star_s,
        "peak_score": peak_s,
        "orig_epsilon_star": orig_eps_star,
        "orig_peak_score": orig_peak
    })

    done_ids.add(s)

    if (s + 1) % 25 == 0:
        partial = pd.DataFrame(rows)
        partial.to_csv(checkpoint_file, index=False)
        partial.to_csv(result_file, index=False)
        print(f"Saved shuffle thresholdress: {s+1}/{N_SHUFFLES}")

# ============================================================
# FINAL SAVE
# ============================================================

shuffle_df = pd.DataFrame(rows)

shuffle_df.to_csv(result_file, index=False)
shuffle_df.to_csv(checkpoint_file, index=False)

# ============================================================
# SUMMARY / P-VALUE
# ============================================================

p_peak = np.mean(shuffle_df["peak_score"] >= orig_peak)
p_same_or_close = np.mean(np.abs(shuffle_df["epsilon_star"] - orig_eps_star) <= 0.02)

summary = pd.DataFrame([{
    "orig_epsilon_star": orig_eps_star,
    "orig_peak_score": orig_peak,
    "shuffle_peak_mean": shuffle_df["peak_score"].mean(),
    "shuffle_peak_std": shuffle_df["peak_score"].std(),
    "shuffle_epsilon_star_mean": shuffle_df["epsilon_star"].mean(),
    "shuffle_epsilon_star_std": shuffle_df["epsilon_star"].std(),
    "p_peak_ge_original": p_peak,
    "p_epsilon_star_within_0p02": p_same_or_close,
    "n_shuffles": len(shuffle_df)
}])

summary.to_csv(summary_file, index=False)

print("\nSaved shuffle result:")
print(result_file)

print("\nSaved summary:")
print(summary_file)

display(summary)

# ============================================================
# PLOT 1 — PEAK SCORE NULL DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    shuffle_df["peak_score"],
    bins=40,
    density=True,
    alpha=0.75
)

plt.axvline(
    orig_peak,
    linestyle="--",
    linewidth=2,
    label=f"original peak={orig_peak:.3f}"
)

plt.xlabel("peak reorganization score")
plt.ylabel("density")
plt.title("CLM v1 — shuffle null distribution of peak score")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_shuffle_peak_score_null_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 2 — EPSILON STAR NULL DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    shuffle_df["epsilon_star"],
    bins=np.arange(-0.01, 1.03, 0.02),
    alpha=0.75
)

plt.axvline(
    orig_eps_star,
    linestyle="--",
    linewidth=2,
    label=f"original ε*={orig_eps_star:.2f}"
)

plt.xlabel("ε*")
plt.ylabel("count")
plt.title("CLM v1 — shuffle null distribution of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_shuffle_epsilon_star_null_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 3 — ORIGINAL SCORE AGAINST SHUFFLE PEAK LEVELS
# ============================================================

plt.figure(figsize=(9,5))

plt.plot(
    eps,
    orig_score,
    marker="o",
    label="original reorganization score"
)

plt.axvline(
    orig_eps_star,
    linestyle="--",
    label=f"ε*={orig_eps_star:.2f}"
)

plt.axhline(
    shuffle_df["peak_score"].mean(),
    linestyle=":",
    label="shuffle peak mean"
)

plt.axhline(
    shuffle_df["peak_score"].quantile(0.95),
    linestyle="-.",
    label="shuffle peak 95%"
)

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — original score vs shuffle null")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_original_vs_shuffle_null_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\nShuffle-control pipeline completed.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — BOOTSTRAP STABILITY OF ε*
# bootstrap metrics / epsilon grid + CSV + plots
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# LOAD ENRICHED DATA
# ============================================================

enriched_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv"))
)[-1]

df = pd.read_csv(enriched_file).sort_values("epsilon").reset_index(drop=True)

print("Loaded:")
print(enriched_file)
print("Rows:", len(df))

# ============================================================
# PARAMETERS
# ============================================================

N_BOOT = 500

METRICS = [
    "sync_score",
    "mean_spatial_var",
    "mean_clusters",
    "mean_switch_fraction",
    "mean_dwell",
    "max_dwell",
    "switching_rate",
    "dominant_fraction",
    "mean_cluster_count"
]

RESULT_FILE = f"clm_v1_epsilon_star_bootstrap_{RUN_ID}.csv"
SUMMARY_FILE = f"clm_v1_epsilon_star_bootstrap_summary_{RUN_ID}.csv"
CHECKPOINT_FILE = "clm_v1_epsilon_star_bootstrap_checkpoint.csv"

result_file = csv_path(RESULT_FILE)
summary_file = csv_path(SUMMARY_FILE)
checkpoint_file = checkpoint_path(CHECKPOINT_FILE)

# ============================================================
# SCORE FUNCTION
# ============================================================

def compute_reorganization_score(input_df, metrics=METRICS):
    temp = input_df.copy().sort_values("epsilon").reset_index(drop=True)
    eps_vals = temp["epsilon"].values

    score_cols = []

    for col in metrics:
        y = temp[col].values.astype(float)
        dy = np.gradient(y, eps_vals)

        abs_dy = np.abs(dy)

        mn = np.nanmin(abs_dy)
        mx = np.nanmax(abs_dy)

        if mx > mn:
            norm = (abs_dy - mn) / (mx - mn)
        else:
            norm = np.zeros_like(abs_dy)

        temp[f"{col}_abs_deriv_norm"] = norm
        score_cols.append(f"{col}_abs_deriv_norm")

    temp["reorganization_score"] = temp[score_cols].mean(axis=1)

    idx = temp["reorganization_score"].idxmax()

    return {
        "epsilon_star": temp.loc[idx, "epsilon"],
        "peak_score": temp.loc[idx, "reorganization_score"],
        "score_curve": temp["reorganization_score"].values
    }

# ============================================================
# ORIGINAL
# ============================================================

orig = compute_reorganization_score(df)

ORIG_EPS_STAR = orig["epsilon_star"]
ORIG_PEAK = orig["peak_score"]

print("Original epsilon_star:", ORIG_EPS_STAR)
print("Original peak:", ORIG_PEAK)

# ============================================================
# RESUME
# ============================================================

if os.path.exists(checkpoint_file):
    boot_df = pd.read_csv(checkpoint_file)
    done = set(boot_df["bootstrap_id"].values)
    rows = boot_df.to_dict("records")

    print("Loaded checkpoint:", checkpoint_file)
    print("Completed bootstrap:", len(done))

else:
    done = set()
    rows = []
    print("No checkpoint found. Starting fresh.")

# ============================================================
# BOOTSTRAP LOOP
# ============================================================

for b in tqdm(range(N_BOOT), desc="bootstrap loop"):

    if b in done:
        continue

    boot = df[["epsilon"] + METRICS].copy()

    # bootstrap metrics: resample metrics with replacement across columns
    chosen_metrics = np.random.choice(
        METRICS,
        size=len(METRICS),
        replace=True
    )

    score_cols = []

    eps_vals = boot["epsilon"].values

    for i, col in enumerate(chosen_metrics):
        y = boot[col].values.astype(float)
        dy = np.gradient(y, eps_vals)
        abs_dy = np.abs(dy)

        mn = np.nanmin(abs_dy)
        mx = np.nanmax(abs_dy)

        if mx > mn:
            norm = (abs_dy - mn) / (mx - mn)
        else:
            norm = np.zeros_like(abs_dy)

        new_col = f"metric_boot_{i}"
        boot[new_col] = norm
        score_cols.append(new_col)

    boot["reorganization_score"] = boot[score_cols].mean(axis=1)

    idx = boot["reorganization_score"].idxmax()

    rows.append({
        "bootstrap_id": b,
        "epsilon_star": boot.loc[idx, "epsilon"],
        "peak_score": boot.loc[idx, "reorganization_score"],
        "orig_epsilon_star": ORIG_EPS_STAR,
        "orig_peak_score": ORIG_PEAK,
        "metrics_used": ",".join(chosen_metrics)
    })

    done.add(b)

    if (b + 1) % 25 == 0:
        partial = pd.DataFrame(rows)
        partial.to_csv(checkpoint_file, index=False)
        partial.to_csv(result_file, index=False)
        print(f"Saved bootstrap thresholdress: {b+1}/{N_BOOT}")

# ============================================================
# FINAL SAVE
# ============================================================

boot_df = pd.DataFrame(rows)

boot_df.to_csv(result_file, index=False)
boot_df.to_csv(checkpoint_file, index=False)

# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame([{
    "orig_epsilon_star": ORIG_EPS_STAR,
    "orig_peak_score": ORIG_PEAK,

    "bootstrap_epsilon_star_mean": boot_df["epsilon_star"].mean(),
    "bootstrap_epsilon_star_median": boot_df["epsilon_star"].median(),
    "bootstrap_epsilon_star_std": boot_df["epsilon_star"].std(),
    "bootstrap_epsilon_star_q05": boot_df["epsilon_star"].quantile(0.05),
    "bootstrap_epsilon_star_q95": boot_df["epsilon_star"].quantile(0.95),

    "bootstrap_peak_mean": boot_df["peak_score"].mean(),
    "bootstrap_peak_std": boot_df["peak_score"].std(),
    "bootstrap_peak_q05": boot_df["peak_score"].quantile(0.05),
    "bootstrap_peak_q95": boot_df["peak_score"].quantile(0.95),

    "p_epsilon_star_within_0p02": np.mean(
        np.abs(boot_df["epsilon_star"] - ORIG_EPS_STAR) <= 0.02
    ),

    "p_epsilon_star_within_0p04": np.mean(
        np.abs(boot_df["epsilon_star"] - ORIG_EPS_STAR) <= 0.04
    ),

    "n_bootstrap": len(boot_df)
}])

summary.to_csv(summary_file, index=False)

print("\nSaved bootstrap result:")
print(result_file)

print("\nSaved bootstrap summary:")
print(summary_file)

display(summary)

# ============================================================
# PLOT 1 — ε* BOOTSTRAP DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=np.arange(-0.01, 1.03, 0.02),
    alpha=0.75
)

plt.axvline(
    ORIG_EPS_STAR,
    linestyle="--",
    linewidth=2,
    label=f"original ε*={ORIG_EPS_STAR:.2f}"
)

plt.xlabel("ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap distribution of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_bootstrap_epsilon_star_distribution_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 2 — PEAK SCORE BOOTSTRAP DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["peak_score"],
    bins=40,
    density=True,
    alpha=0.75
)

plt.axvline(
    ORIG_PEAK,
    linestyle="--",
    linewidth=2,
    label=f"original peak={ORIG_PEAK:.3f}"
)

plt.xlabel("peak score")
plt.ylabel("density")
plt.title("CLM v1 — bootstrap peak-score distribution")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_bootstrap_peak_score_distribution_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 3 — ε* FREQUENCY BAR
# ============================================================

freq = (
    boot_df["epsilon_star"]
    .value_counts()
    .sort_index()
    .reset_index()
)

freq.columns = ["epsilon_star", "count"]

plt.figure(figsize=(10,5))

plt.bar(
    freq["epsilon_star"],
    freq["count"],
    width=0.015
)

plt.axvline(
    ORIG_EPS_STAR,
    linestyle="--",
    linewidth=2,
    label=f"original ε*={ORIG_EPS_STAR:.2f}"
)

plt.xlabel("ε*")
plt.ylabel("count")
plt.title("CLM v1 — ε* bootstrap frequency")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_bootstrap_epsilon_star_frequency_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\nBootstrap stability pipeline completed.")

In [ ]:
# ============================================================
# CLM v1 — BOOTSTRAP ε* WITHOUT ENDPOINT EFFECTS
# epsilon range restricted to remove artificial endpoint peak
# CSV + plots + checkpoint
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================
# LOAD ENRICHED DATA
# ============================================================

enriched_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv"))
)[-1]

df0 = pd.read_csv(enriched_file).sort_values("epsilon").reset_index(drop=True)

# remove endpoint region
EPS_MIN = 0.02
EPS_MAX = 0.90

df = df0[
    (df0["epsilon"] >= EPS_MIN) &
    (df0["epsilon"] <= EPS_MAX)
].copy().reset_index(drop=True)

print("Loaded:")
print(enriched_file)
print("Original rows:", len(df0))
print("Filtered rows:", len(df))
print("Epsilon range:", df["epsilon"].min(), "to", df["epsilon"].max())

# ============================================================
# PARAMETERS
# ============================================================

N_BOOT = 500

METRICS = [
    "sync_score",
    "mean_spatial_var",
    "mean_clusters",
    "mean_switch_fraction",
    "mean_dwell",
    "max_dwell",
    "switching_rate",
    "dominant_fraction",
    "mean_cluster_count"
]

RESULT_FILE = f"clm_v1_epsilon_star_bootstrap_no_endpoint_{RUN_ID}.csv"
SUMMARY_FILE = f"clm_v1_epsilon_star_bootstrap_no_endpoint_summary_{RUN_ID}.csv"
CHECKPOINT_FILE = "clm_v1_epsilon_star_bootstrap_no_endpoint_checkpoint.csv"

result_file = csv_path(RESULT_FILE)
summary_file = csv_path(SUMMARY_FILE)
checkpoint_file = checkpoint_path(CHECKPOINT_FILE)

# ============================================================
# SCORE FUNCTION
# ============================================================

def compute_reorganization_score(input_df, metrics=METRICS):
    temp = input_df.copy().sort_values("epsilon").reset_index(drop=True)
    eps_vals = temp["epsilon"].values

    score_cols = []

    for col in metrics:
        y = temp[col].values.astype(float)
        dy = np.gradient(y, eps_vals)
        abs_dy = np.abs(dy)

        mn = np.nanmin(abs_dy)
        mx = np.nanmax(abs_dy)

        if mx > mn:
            norm = (abs_dy - mn) / (mx - mn)
        else:
            norm = np.zeros_like(abs_dy)

        new_col = f"{col}_abs_deriv_norm"
        temp[new_col] = norm
        score_cols.append(new_col)

    temp["reorganization_score"] = temp[score_cols].mean(axis=1)

    idx = temp["reorganization_score"].idxmax()

    return {
        "epsilon_star": temp.loc[idx, "epsilon"],
        "peak_score": temp.loc[idx, "reorganization_score"],
        "score_curve": temp["reorganization_score"].values
    }

# ============================================================
# ORIGINAL FILTERED
# ============================================================

orig = compute_reorganization_score(df)

ORIG_EPS_STAR = orig["epsilon_star"]
ORIG_PEAK = orig["peak_score"]
ORIG_SCORE = orig["score_curve"]

print("Filtered original epsilon_star:", ORIG_EPS_STAR)
print("Filtered original peak:", ORIG_PEAK)

# ============================================================
# RESUME
# ============================================================

if os.path.exists(checkpoint_file):
    boot_df = pd.read_csv(checkpoint_file)
    done = set(boot_df["bootstrap_id"].values)
    rows = boot_df.to_dict("records")

    print("Loaded checkpoint:", checkpoint_file)
    print("Completed bootstrap:", len(done))

else:
    done = set()
    rows = []
    print("No checkpoint found. Starting fresh.")

# ============================================================
# BOOTSTRAP LOOP
# ============================================================

for b in tqdm(range(N_BOOT), desc="bootstrap no-endpoint loop"):

    if b in done:
        continue

    boot = df[["epsilon"] + METRICS].copy()

    chosen_metrics = np.random.choice(
        METRICS,
        size=len(METRICS),
        replace=True
    )

    eps_vals = boot["epsilon"].values
    score_cols = []

    for i, col in enumerate(chosen_metrics):
        y = boot[col].values.astype(float)
        dy = np.gradient(y, eps_vals)
        abs_dy = np.abs(dy)

        mn = np.nanmin(abs_dy)
        mx = np.nanmax(abs_dy)

        if mx > mn:
            norm = (abs_dy - mn) / (mx - mn)
        else:
            norm = np.zeros_like(abs_dy)

        new_col = f"metric_boot_{i}"
        boot[new_col] = norm
        score_cols.append(new_col)

    boot["reorganization_score"] = boot[score_cols].mean(axis=1)

    idx = boot["reorganization_score"].idxmax()

    rows.append({
        "bootstrap_id": b,
        "epsilon_star": boot.loc[idx, "epsilon"],
        "peak_score": boot.loc[idx, "reorganization_score"],
        "orig_epsilon_star": ORIG_EPS_STAR,
        "orig_peak_score": ORIG_PEAK,
        "eps_min": EPS_MIN,
        "eps_max": EPS_MAX,
        "metrics_used": ",".join(chosen_metrics)
    })

    done.add(b)

    if (b + 1) % 25 == 0:
        partial = pd.DataFrame(rows)
        partial.to_csv(checkpoint_file, index=False)
        partial.to_csv(result_file, index=False)
        print(f"Saved bootstrap thresholdress: {b+1}/{N_BOOT}")

# ============================================================
# FINAL SAVE
# ============================================================

boot_df = pd.DataFrame(rows)

boot_df.to_csv(result_file, index=False)
boot_df.to_csv(checkpoint_file, index=False)

# ============================================================
# SUMMARY
# ============================================================

summary = pd.DataFrame([{
    "eps_min": EPS_MIN,
    "eps_max": EPS_MAX,

    "orig_epsilon_star": ORIG_EPS_STAR,
    "orig_peak_score": ORIG_PEAK,

    "bootstrap_epsilon_star_mean": boot_df["epsilon_star"].mean(),
    "bootstrap_epsilon_star_median": boot_df["epsilon_star"].median(),
    "bootstrap_epsilon_star_std": boot_df["epsilon_star"].std(),
    "bootstrap_epsilon_star_q05": boot_df["epsilon_star"].quantile(0.05),
    "bootstrap_epsilon_star_q95": boot_df["epsilon_star"].quantile(0.95),

    "bootstrap_peak_mean": boot_df["peak_score"].mean(),
    "bootstrap_peak_std": boot_df["peak_score"].std(),
    "bootstrap_peak_q05": boot_df["peak_score"].quantile(0.05),
    "bootstrap_peak_q95": boot_df["peak_score"].quantile(0.95),

    "p_epsilon_star_within_0p02": np.mean(
        np.abs(boot_df["epsilon_star"] - ORIG_EPS_STAR) <= 0.02
    ),

    "p_epsilon_star_within_0p04": np.mean(
        np.abs(boot_df["epsilon_star"] - ORIG_EPS_STAR) <= 0.04
    ),

    "n_bootstrap": len(boot_df)
}])

summary.to_csv(summary_file, index=False)

print("\nSaved bootstrap result:")
print(result_file)

print("\nSaved bootstrap summary:")
print(summary_file)

display(summary)

# ============================================================
# PLOT 1 — ε* DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=np.arange(EPS_MIN - 0.01, EPS_MAX + 0.03, 0.02),
    alpha=0.75
)

plt.axvline(
    ORIG_EPS_STAR,
    linestyle="--",
    linewidth=2,
    label=f"filtered original ε*={ORIG_EPS_STAR:.2f}"
)

plt.xlabel("ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap ε* without endpoints")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_bootstrap_epsilon_star_no_endpoint_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 2 — PEAK SCORE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["peak_score"],
    bins=40,
    density=True,
    alpha=0.75
)

plt.axvline(
    ORIG_PEAK,
    linestyle="--",
    linewidth=2,
    label=f"filtered original peak={ORIG_PEAK:.3f}"
)

plt.xlabel("peak score")
plt.ylabel("density")
plt.title("CLM v1 — bootstrap peak score without endpoints")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_bootstrap_peak_score_no_endpoint_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# PLOT 3 — FILTERED ORIGINAL SCORE
# ============================================================

plt.figure(figsize=(9,5))

plt.plot(
    df["epsilon"],
    ORIG_SCORE,
    marker="o",
    label="filtered original score"
)

plt.axvline(
    ORIG_EPS_STAR,
    linestyle="--",
    label=f"ε*={ORIG_EPS_STAR:.2f}"
)

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — filtered original reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = fig_path(f"clm_v1_filtered_reorganization_score_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\nNo-endpoint bootstrap completed.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1 — MASTER SUMMARY + FINAL FIGURES
# collects main / dwell / reorganization / shuffle / bootstrap
# ============================================================

import glob
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD LATEST FILES
# ============================================================

main_file = sorted(glob.glob(csv_path("clm_v1_main_results_*.csv")))[-1]
dwell_file = sorted(glob.glob(csv_path("clm_v1_cluster_dwell_summary_*.csv")))[-1]
reorg_file = sorted(glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv")))[-1]
reorg_summary_file = sorted(glob.glob(csv_path("clm_v1_reorganization_summary_*.csv")))[-1]
shuffle_summary_file = sorted(glob.glob(csv_path("clm_v1_reorganization_shuffle_summary_*.csv")))[-1]
boot_summary_file = sorted(glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_summary_*.csv")))[-1]

main_df = pd.read_csv(main_file)
dwell_df = pd.read_csv(dwell_file)
reorg_df = pd.read_csv(reorg_file)
reorg_summary = pd.read_csv(reorg_summary_file)
shuffle_summary = pd.read_csv(shuffle_summary_file)
boot_summary = pd.read_csv(boot_summary_file)

# ============================================================
# MASTER MERGE
# ============================================================

master = main_df.merge(
    dwell_df,
    on="epsilon",
    how="left",
    suffixes=("_main", "_cluster")
)

master = master.merge(
    reorg_df[["epsilon", "reorganization_score"]],
    on="epsilon",
    how="left"
)

master_path = csv_path(f"clm_v1_MASTER_SUMMARY_{RUN_ID}.csv")
master.to_csv(master_path, index=False)

print("Saved MASTER:")
print(master_path)
print("Rows:", len(master))

display(master.head())

# ============================================================
# SAVE ONE-LINE SUMMARY
# ============================================================

one_line = pd.concat(
    [
        reorg_summary.reset_index(drop=True),
        shuffle_summary.reset_index(drop=True),
        boot_summary.reset_index(drop=True)
    ],
    axis=1
)

one_line_path = csv_path(f"clm_v1_FINAL_NUMERIC_SUMMARY_{RUN_ID}.csv")
one_line.to_csv(one_line_path, index=False)

print("Saved FINAL NUMERIC SUMMARY:")
print(one_line_path)

display(one_line)

# ============================================================
# FINAL FIGURES
# ============================================================

# ------------------------------------------------------------
# FIG 1 — reorganization score
# ------------------------------------------------------------

eps_star = reorg_summary.loc[0, "epsilon_star"]
wmin = reorg_summary.loc[0, "epsilon_window_min"]
wmax = reorg_summary.loc[0, "epsilon_window_max"]

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["reorganization_score"],
    marker="o"
)

plt.axvline(eps_star, linestyle="--", label=f"ε*={eps_star:.2f}")
plt.axvspan(wmin, wmax, alpha=0.2, label="window")

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — reorganization window")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG1_clm_v1_reorganization_window_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# FIG 2 — clusters + dwell
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["mean_clusters"],
    marker="o",
    label="mean clusters"
)

plt.plot(
    master["epsilon"],
    master["mean_dwell"],
    marker="o",
    label="cluster dwell"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — clusters and dwell")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG2_clm_v1_clusters_dwell_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# FIG 3 — synchronization and variance
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["sync_score"],
    marker="o",
    label="sync score"
)

plt.plot(
    master["epsilon"],
    master["mean_spatial_var"],
    marker="o",
    label="spatial variance"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — synchronization / variance")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG3_clm_v1_sync_variance_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# FIG 4 — switching and dominant fraction
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["switching_rate"],
    marker="o",
    label="cluster switching rate"
)

plt.plot(
    master["epsilon"],
    master["dominant_fraction"],
    marker="o",
    label="dominant fraction"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — switching and memory")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG4_clm_v1_switching_memory_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ------------------------------------------------------------
# FIG 5 — bootstrap no-endpoint ε*
# ------------------------------------------------------------

boot_file = sorted(glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_*.csv")))[-1]
boot_df = pd.read_csv(boot_file)

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=30,
    alpha=0.75
)

plt.axvline(eps_star, linestyle="--", label=f"ε*={eps_star:.2f}")

plt.xlabel("bootstrap ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap stability of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG5_clm_v1_bootstrap_epsilon_star_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\nCLM v1 MASTER SUMMARY + FINAL FIGURES COMPLETED.")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1
# MASTER SUMMARY + FINAL FIGURES
# FULL FIXED VERSION
# ============================================================

import glob
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD LATEST FILES
# ============================================================

main_file = sorted(
    glob.glob(csv_path("clm_v1_main_results_*.csv"))
)[-1]

dwell_file = sorted(
    glob.glob(csv_path("clm_v1_cluster_dwell_summary_*.csv"))
)[-1]

reorg_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv"))
)[-1]

reorg_summary_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_summary_*.csv"))
)[-1]

shuffle_summary_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_shuffle_summary_*.csv"))
)[-1]

boot_summary_file = sorted(
    glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_summary_*.csv"))
)[-1]

boot_file = sorted(
    glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_*.csv"))
)[-1]

# ============================================================
# READ CSV
# ============================================================

main_df = pd.read_csv(main_file)
dwell_df = pd.read_csv(dwell_file)
reorg_df = pd.read_csv(reorg_file)

reorg_summary = pd.read_csv(reorg_summary_file)
shuffle_summary = pd.read_csv(shuffle_summary_file)
boot_summary = pd.read_csv(boot_summary_file)

boot_df = pd.read_csv(boot_file)

# ============================================================
# DEBUG INFO
# ============================================================

print("\nREORG SUMMARY COLUMNS:")
print(reorg_summary.columns)

print("\nBOOT SUMMARY COLUMNS:")
print(boot_summary.columns)

# ============================================================
# EXTRACT WINDOW PARAMETERS
# ============================================================

eps_star = reorg_summary.loc[0, "epsilon_star"]
wmin = reorg_summary.loc[0, "epsilon_window_min"]
wmax = reorg_summary.loc[0, "epsilon_window_max"]

# ============================================================
# MASTER MERGE
# ============================================================

master = main_df.merge(
    dwell_df,
    on="epsilon",
    how="left",
    suffixes=("_main", "_cluster")
)

master = master.merge(
    reorg_df[["epsilon", "reorganization_score"]],
    on="epsilon",
    how="left"
)

# ============================================================
# SAVE MASTER SUMMARY
# ============================================================

master_path = csv_path(
    f"clm_v1_MASTER_SUMMARY_{RUN_ID}.csv"
)

master.to_csv(master_path, index=False)

print("\nSaved MASTER SUMMARY:")
print(master_path)

print("\nMASTER rows:", len(master))

display(master.head())

# ============================================================
# FINAL NUMERIC SUMMARY
# ============================================================

final_summary = pd.DataFrame({
    "epsilon_star": [eps_star],
    "window_min": [wmin],
    "window_max": [wmax],

    "peak_score": [
        reorg_summary.loc[0, "reorganization_score_peak"]
    ],

    "shuffle_peak_mean": [
        shuffle_summary.loc[0, "shuffle_peak_mean"]
    ],

    "shuffle_peak_std": [
        shuffle_summary.loc[0, "shuffle_peak_std"]
    ],

    "shuffle_p_ge_original": [
        shuffle_summary.loc[0, "p_peak_ge_original"]
    ],

    "bootstrap_eps_mean": [
        boot_summary.loc[0, "bootstrap_epsilon_star_mean"]
    ],

    "bootstrap_eps_median": [
        boot_summary.loc[0, "bootstrap_epsilon_star_median"]
    ],

    "bootstrap_eps_std": [
        boot_summary.loc[0, "bootstrap_epsilon_star_std"]
    ],

    "bootstrap_eps_q05": [
        boot_summary.loc[0, "bootstrap_epsilon_star_q05"]
    ],

    "bootstrap_eps_q95": [
        boot_summary.loc[0, "bootstrap_epsilon_star_q95"]
    ],

    "bootstrap_peak_mean": [
        boot_summary.loc[0, "bootstrap_peak_mean"]
    ],

    "bootstrap_peak_std": [
        boot_summary.loc[0, "bootstrap_peak_std"]
    ],

    "bootstrap_p_within_002": [
        boot_summary.loc[0, "p_epsilon_star_within_0p02"]
    ],

    "bootstrap_p_within_004": [
        boot_summary.loc[0, "p_epsilon_star_within_0p04"]
    ]
})

summary_path = csv_path(
    f"clm_v1_FINAL_NUMERIC_SUMMARY_{RUN_ID}.csv"
)

final_summary.to_csv(summary_path, index=False)

print("\nSaved FINAL NUMERIC SUMMARY:")
print(summary_path)

display(final_summary)

# ============================================================
# FINAL FIGURE 1
# REORGANIZATION WINDOW
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["reorganization_score"],
    marker="o"
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.axvspan(
    wmin,
    wmax,
    alpha=0.2,
    label="window"
)

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — reorganization window")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(
    f"FIG1_clm_v1_reorganization_window_{RUN_ID}.png"
)

plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINAL FIGURE 2
# CLUSTERS + DWELL
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["mean_clusters"],
    marker="o",
    label="mean clusters"
)

plt.plot(
    master["epsilon"],
    master["mean_dwell"],
    marker="o",
    label="cluster dwell"
)

plt.axvspan(
    wmin,
    wmax,
    alpha=0.2
)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — clusters and dwell")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(
    f"FIG2_clm_v1_clusters_dwell_{RUN_ID}.png"
)

plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINAL FIGURE 3
# SYNC + VARIANCE
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["sync_score"],
    marker="o",
    label="sync score"
)

plt.plot(
    master["epsilon"],
    master["mean_spatial_var"],
    marker="o",
    label="spatial variance"
)

plt.axvspan(
    wmin,
    wmax,
    alpha=0.2
)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — synchronization and variance")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(
    f"FIG3_clm_v1_sync_variance_{RUN_ID}.png"
)

plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINAL FIGURE 4
# SWITCHING + MEMORY
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["switching_rate"],
    marker="o",
    label="switching rate"
)

plt.plot(
    master["epsilon"],
    master["dominant_fraction"],
    marker="o",
    label="dominant fraction"
)

plt.axvspan(
    wmin,
    wmax,
    alpha=0.2
)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — switching and memory")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(
    f"FIG4_clm_v1_switching_memory_{RUN_ID}.png"
)

plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINAL FIGURE 5
# BOOTSTRAP EPSILON STAR
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=30,
    alpha=0.75
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.xlabel("bootstrap ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap stability of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(
    f"FIG5_clm_v1_bootstrap_epsilon_star_{RUN_ID}.png"
)

plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINISHED
# ============================================================

print("\n===================================================")
print("CLM v1 MASTER SUMMARY + FINAL FIGURES COMPLETED.")
print("===================================================")

In [ ]:
# ============================================================
# COUPLED LOGISTIC MAPS v1
# MASTER SUMMARY + FINAL FIGURES
# SAFE FIXED VERSION
# ============================================================

import glob
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD LATEST FILES
# ============================================================

main_file = sorted(glob.glob(csv_path("clm_v1_main_results_*.csv")))[-1]
dwell_file = sorted(glob.glob(csv_path("clm_v1_cluster_dwell_summary_*.csv")))[-1]
reorg_file = sorted(glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv")))[-1]
reorg_summary_file = sorted(glob.glob(csv_path("clm_v1_reorganization_summary_*.csv")))[-1]
shuffle_summary_file = sorted(glob.glob(csv_path("clm_v1_reorganization_shuffle_summary_*.csv")))[-1]
boot_summary_file = sorted(glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_summary_*.csv")))[-1]
boot_file = sorted(glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_*.csv")))[-1]

main_df = pd.read_csv(main_file)
dwell_df = pd.read_csv(dwell_file)
reorg_df = pd.read_csv(reorg_file)
reorg_summary = pd.read_csv(reorg_summary_file)
shuffle_summary = pd.read_csv(shuffle_summary_file)
boot_summary = pd.read_csv(boot_summary_file)
boot_df = pd.read_csv(boot_file)

print("Loaded files:")
print(main_file)
print(dwell_file)
print(reorg_file)
print(reorg_summary_file)
print(shuffle_summary_file)
print(boot_summary_file)
print(boot_file)

# ============================================================
# SAFE HELPERS
# ============================================================

def get_first_existing(df, names, default=None):
    for name in names:
        if name in df.columns:
            return df.loc[0, name]
    return default

# ============================================================
# SAFE WINDOW EXTRACTION
# ============================================================

print("\nREORG SUMMARY COLUMNS:")
print(reorg_summary.columns.tolist())

print("\nSHUFFLE SUMMARY COLUMNS:")
print(shuffle_summary.columns.tolist())

print("\nBOOT SUMMARY COLUMNS:")
print(boot_summary.columns.tolist())

eps_star = get_first_existing(
    reorg_summary,
    ["epsilon_star", "orig_epsilon_star"],
    default=None
)

if eps_star is None:
    raise ValueError("No epsilon_star / orig_epsilon_star column found.")

wmin = get_first_existing(
    reorg_summary,
    ["epsilon_window_min", "window_min"],
    default=eps_star
)

wmax = get_first_existing(
    reorg_summary,
    ["epsilon_window_max", "window_max"],
    default=eps_star
)

peak_score = get_first_existing(
    reorg_summary,
    ["reorganization_score_peak", "orig_peak_score", "peak_score"],
    default=None
)

print("\nUSING:")
print("eps_star:", eps_star)
print("window:", wmin, "->", wmax)
print("peak_score:", peak_score)

# ============================================================
# MASTER MERGE
# ============================================================

master = main_df.merge(
    dwell_df,
    on="epsilon",
    how="left",
    suffixes=("_main", "_cluster")
)

if "reorganization_score" in reorg_df.columns:
    master = master.merge(
        reorg_df[["epsilon", "reorganization_score"]],
        on="epsilon",
        how="left"
    )
else:
    print("WARNING: no reorganization_score column in reorg_df")

master_path = csv_path(f"clm_v1_MASTER_SUMMARY_{RUN_ID}.csv")
master.to_csv(master_path, index=False)

print("\nSaved MASTER SUMMARY:")
print(master_path)
print("Rows:", len(master))

display(master.head())

# ============================================================
# FINAL NUMERIC SUMMARY — SAFE
# ============================================================

final_summary = pd.DataFrame([{
    "epsilon_star": eps_star,
    "window_min": wmin,
    "window_max": wmax,
    "peak_score": peak_score,

    "shuffle_peak_mean": get_first_existing(
        shuffle_summary,
        ["shuffle_peak_mean"],
        default=None
    ),

    "shuffle_peak_std": get_first_existing(
        shuffle_summary,
        ["shuffle_peak_std"],
        default=None
    ),

    "shuffle_p_ge_original": get_first_existing(
        shuffle_summary,
        ["p_peak_ge_original"],
        default=None
    ),

    "shuffle_epsilon_star_mean": get_first_existing(
        shuffle_summary,
        ["shuffle_epsilon_star_mean"],
        default=None
    ),

    "shuffle_epsilon_star_std": get_first_existing(
        shuffle_summary,
        ["shuffle_epsilon_star_std"],
        default=None
    ),

    "bootstrap_eps_mean": get_first_existing(
        boot_summary,
        ["bootstrap_epsilon_star_mean"],
        default=None
    ),

    "bootstrap_eps_median": get_first_existing(
        boot_summary,
        ["bootstrap_epsilon_star_median"],
        default=None
    ),

    "bootstrap_eps_std": get_first_existing(
        boot_summary,
        ["bootstrap_epsilon_star_std"],
        default=None
    ),

    "bootstrap_eps_q05": get_first_existing(
        boot_summary,
        ["bootstrap_epsilon_star_q05"],
        default=None
    ),

    "bootstrap_eps_q95": get_first_existing(
        boot_summary,
        ["bootstrap_epsilon_star_q95"],
        default=None
    ),

    "bootstrap_peak_mean": get_first_existing(
        boot_summary,
        ["bootstrap_peak_mean"],
        default=None
    ),

    "bootstrap_peak_std": get_first_existing(
        boot_summary,
        ["bootstrap_peak_std"],
        default=None
    ),

    "bootstrap_p_within_002": get_first_existing(
        boot_summary,
        ["p_epsilon_star_within_0p02"],
        default=None
    ),

    "bootstrap_p_within_004": get_first_existing(
        boot_summary,
        ["p_epsilon_star_within_0p04"],
        default=None
    )
}])

summary_path = csv_path(f"clm_v1_FINAL_NUMERIC_SUMMARY_{RUN_ID}.csv")
final_summary.to_csv(summary_path, index=False)

print("\nSaved FINAL NUMERIC SUMMARY:")
print(summary_path)

display(final_summary)

# ============================================================
# FINAL FIGURE 1 — REORGANIZATION WINDOW
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["reorganization_score"],
    marker="o",
    label="reorganization score"
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.axvspan(
    wmin,
    wmax,
    alpha=0.2,
    label="window"
)

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — reorganization window")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG1_clm_v1_reorganization_window_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FINAL FIGURE 2 — CLUSTERS + DWELL
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["mean_clusters"],
    marker="o",
    label="mean clusters"
)

plt.plot(
    master["epsilon"],
    master["mean_dwell"],
    marker="o",
    label="cluster dwell"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — clusters and dwell")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG2_clm_v1_clusters_dwell_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FINAL FIGURE 3 — SYNC + VARIANCE
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["sync_score"],
    marker="o",
    label="sync score"
)

plt.plot(
    master["epsilon"],
    master["mean_spatial_var"],
    marker="o",
    label="spatial variance"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — synchronization and variance")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG3_clm_v1_sync_variance_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FINAL FIGURE 4 — SWITCHING + MEMORY
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    master["epsilon"],
    master["switching_rate"],
    marker="o",
    label="cluster switching rate"
)

plt.plot(
    master["epsilon"],
    master["dominant_fraction"],
    marker="o",
    label="dominant fraction"
)

plt.axvspan(wmin, wmax, alpha=0.2)

plt.xlabel("coupling ε")
plt.ylabel("value")
plt.title("CLM v1 — switching and memory")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG4_clm_v1_switching_memory_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

# ============================================================
# FINAL FIGURE 5 — BOOTSTRAP EPSILON STAR
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=30,
    alpha=0.75
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.xlabel("bootstrap ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap stability of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG5_clm_v1_bootstrap_epsilon_star_{RUN_ID}.png")
plt.savefig(fname, dpi=300)
print("Saved:", fname)
plt.show()

print("\n===================================================")
print("CLM v1 MASTER SUMMARY + FINAL FIGURES COMPLETED.")
print("===================================================")

In [ ]:
# ============================================================
# CLM v1 — FINAL FIGURE PACK (FIXED VERSION)
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "data/raw/delta_window_coupled_logistic_maps_v1"

CSV_DIR = os.path.join(BASE_DIR, "csv")
FINAL_FIG_DIR = os.path.join(BASE_DIR, "final_figures")

os.makedirs(FINAL_FIG_DIR, exist_ok=True)

RUN_ID = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

def csv_path(pattern):
    return os.path.join(CSV_DIR, pattern)

def final_fig_path(name):
    return os.path.join(FINAL_FIG_DIR, name)

# ============================================================
# LOAD FILES
# ============================================================

main_file = sorted(
    glob.glob(csv_path("clm_v1_main_results_*.csv"))
)[-1]

dwell_file = sorted(
    glob.glob(csv_path("clm_v1_cluster_dwell_summary_*.csv"))
)[-1]

reorg_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_enriched_*.csv"))
)[-1]

reorg_summary_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_summary_*.csv"))
)[-1]

shuffle_summary_file = sorted(
    glob.glob(csv_path("clm_v1_reorganization_shuffle_summary_*.csv"))
)[-1]

boot_summary_file = sorted(
    glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_summary_*.csv"))
)[-1]

# IMPORTANT FIX:
boot_raw_files = [
    f for f in glob.glob(csv_path("clm_v1_epsilon_star_bootstrap_no_endpoint_*.csv"))
    if "summary" not in f
]

boot_raw_file = sorted(boot_raw_files)[-1]

# ============================================================
# READ CSV
# ============================================================

main_df = pd.read_csv(main_file)
dwell_df = pd.read_csv(dwell_file)
reorg_df = pd.read_csv(reorg_file)

reorg_summary = pd.read_csv(reorg_summary_file)
shuffle_summary = pd.read_csv(shuffle_summary_file)
boot_summary = pd.read_csv(boot_summary_file)

# FIXED RAW BOOTSTRAP
boot_df = pd.read_csv(boot_raw_file)

print("Loaded raw bootstrap:")
print(boot_raw_file)
print("Columns:", boot_df.columns.tolist())

# ============================================================
# EXTRACT VALUES
# ============================================================

eps_star = reorg_summary.loc[0, "epsilon_star"]
window_min = reorg_summary.loc[0, "epsilon_window_min"]
window_max = reorg_summary.loc[0, "epsilon_window_max"]

# ============================================================
# FIG 1 — REORGANIZATION WINDOW
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    reorg_df["epsilon"],
    reorg_df["reorganization_score"],
    marker="o",
    linewidth=2
)

plt.axvspan(
    window_min,
    window_max,
    alpha=0.25
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.xlabel("coupling ε")
plt.ylabel("reorganization score")
plt.title("CLM v1 — reorganization window")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG1_clm_v1_reorganization_window_{RUN_ID}.png")
plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FIG 2 — CLUSTERS + DWELL
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    dwell_df["epsilon"],
    dwell_df["mean_cluster_count"],
    marker="o",
    label="mean clusters"
)

plt.plot(
    dwell_df["epsilon"],
    dwell_df["mean_dwell"],
    marker="o",
    label="mean dwell"
)

plt.axvline(
    eps_star,
    linestyle="--"
)

plt.xlabel("coupling ε")
plt.ylabel("metric")
plt.title("CLM v1 — clusters and dwell")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG2_clm_v1_clusters_dwell_{RUN_ID}.png")
plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FIG 3 — SYNC + VARIANCE
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    main_df["epsilon"],
    main_df["sync_score"],
    marker="o",
    label="sync score"
)

plt.plot(
    main_df["epsilon"],
    main_df["mean_spatial_var"],
    marker="o",
    label="spatial variance"
)

plt.axvline(
    eps_star,
    linestyle="--"
)

plt.xlabel("coupling ε")
plt.ylabel("metric")
plt.title("CLM v1 — synchronization and variance")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG3_clm_v1_sync_variance_{RUN_ID}.png")
plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FIG 4 — SWITCHING + MEMORY
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    dwell_df["epsilon"],
    dwell_df["switching_rate"],
    marker="o",
    label="switching rate"
)

plt.plot(
    dwell_df["epsilon"],
    dwell_df["dominant_fraction"],
    marker="o",
    label="dominant fraction"
)

plt.axvline(
    eps_star,
    linestyle="--"
)

plt.xlabel("coupling ε")
plt.ylabel("metric")
plt.title("CLM v1 — switching and memory")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG4_clm_v1_switching_memory_{RUN_ID}.png")
plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FIG 5 — FIXED BOOTSTRAP EPSILON STAR
# ============================================================

plt.figure(figsize=(8,5))

plt.hist(
    boot_df["epsilon_star"],
    bins=30,
    alpha=0.75
)

plt.axvline(
    eps_star,
    linestyle="--",
    label=f"ε*={eps_star:.2f}"
)

plt.xlabel("bootstrap ε*")
plt.ylabel("count")
plt.title("CLM v1 — bootstrap stability of ε*")
plt.legend()
plt.grid(True)
plt.tight_layout()

fname = final_fig_path(f"FIG5_clm_v1_bootstrap_epsilon_star_{RUN_ID}.png")
plt.savefig(fname, dpi=300)

print("Saved:", fname)

plt.show()

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\nAll final figures completed successfully.")
print("Output directory:")
print(FINAL_FIG_DIR)